<a href="https://colab.research.google.com/github/andrellyrichelly-hub/ATIVIDADE-PROFESSOR-CLAUDOMIRO_-ALUNA-DE-IA-ANDRELLY-UFPA/blob/main/ATIVI_2_5_3_MEMORIA_ANDRELLY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Atividade 2.5 - Memória 3: Chip DRAM 128M x 4 - 512 Mbits, 2 Bancos
# Andrelly Richelly - Arquitetura com Claudomiro

class ChipDRAM_128Mx4_2Bancos:
    """
    Simula um chip DRAM 128M x 4 bits = 512 Mbits, com 2 bancos
    Endereçamento: 13 bits A0-A12 para linha/coluna + 1 bit de banco
    Total: 2^27 = 134.217.728 posições de 4 bits cada
    """
    def __init__(self):
        self.num_bancos = 2
        self.linhas = 2**13 # 8192 linhas - A0 a A12
        self.colunas = 2**13 # 8192 colunas - A0 a A12
        self.largura = 4 # 4 bits por posição: D0-D3
        self.total_posicoes = 2**27 # 128M posições
        # Dicionário pra economizar RAM: só guarda nibble!= 0
        self.mem = {0: {}, 1: {}}

    def decodificar_endereco(self, endereco_27bits, banco):
        """
        Endereço de 27 bits: [Banco | Linha 12-0 | Coluna 12-0]
        Bit 26 = seleção de banco
        Bits 25-13 = linha, Bits 12-0 = coluna
        """
        if not 0 <= endereco_27bits < self.total_posicoes:
            raise ValueError(f"Endereço {endereco_27bits} fora do limite 0-{self.total_posicoes-1}")
        if banco not in [0, 1]:
            raise ValueError("Banco deve ser 0 ou 1")

        linha = (endereco_27bits >> 13) & 0x1FFF # bits 25-13
        coluna = endereco_27bits & 0x1FFF # bits 12-0
        return linha, coluna

    def ler(self, endereco, banco, cs=0, oe=0, ras=0, cas=0):
        """
        Leitura: CS=0, WE=1, OE=0. Retorna nibble de 4 bits
        """
        if cs == 0 and oe == 0:
            linha, coluna = self.decodificar_endereco(endereco, banco)
            dado = self.mem[banco].get((linha, coluna), 0) # 0 a 15
            print(f"LEITURA: Banco={banco} | End={endereco:08d} | Lin={linha:04d} | Col={coluna:04d} | D={dado:01X} | {dado:04b}")
            return dado
        else:
            print(f"LEITURA BLOQUEADA: CS={cs} OE={oe}")
            return None

    def escrever(self, endereco, banco, dado, cs=0, we=0, ras=0, cas=0):
        """
        Escrita: CS=0, WE=0. Dado é nibble de 4 bits: 0 a 15
        """
        if cs == 0 and we == 0:
            linha, coluna = self.decodificar_endereco(endereco, banco)
            dado_nibble = dado & 0xF # Garante 4 bits
            if dado_nibble:
                self.mem[banco][(linha, coluna)] = dado_nibble
            else:
                self.mem[banco].pop((linha, coluna), None)
            print(f"ESCRITA: Banco={banco} | End={endereco:08d} | Lin={linha:04d} | Col={coluna:04d} | D={dado_nibble:01X} | {dado_nibble:04b}")
        else:
            print(f"ESCRITA BLOQUEADA: CS={cs} WE={we}")

# --- TESTE AUTOMÁTICO - RODA SOZINHO NO COLAB ---
print("=== ATIVIDADE 2.5 - MEMÓRIA 3: DRAM 128M x 4, 2 BANCOS ===\n")

dram4 = ChipDRAM_128Mx4_2Bancos()
print(f"Chip inicializado: 2 bancos, {dram4.linhas}x{dram4.colunas} cada")
print(f"Total: {dram4.total_posicoes} posições de 4 bits = {dram4.total_posicoes * 4 / 8 / 1024 / 1024:.0f} MBytes = 512 Mbits\n")

print("Testando escrita de nibbles:")
dram4.escrever(0, banco=0, dado=0xA) # End 0, Banco 0, Dado 1010
dram4.escrever(8192, banco=0, dado=0xF) # Linha 1, Col 0, Banco 0
dram4.escrever(0, banco=1, dado=0x5) # End 0, Banco 1, Dado 0101
dram4.escrever(134217727, banco=1, dado=0x3) # Última posição, Banco 1

print("\nTestando leitura:")
dram4.ler(0, banco=0, cs=0, oe=0)
dram4.ler(8192, banco=0, cs=0, oe=0)
dram4.ler(0, banco=1, cs=0, oe=0)
dram4.ler(134217727, banco=1, cs=0, oe=0)
dram4.ler(100, banco=0, cs=0, oe=0) # nunca escrito, deve ser 0

print("\nTeste de sinais de controle:")
dram4.ler(0, banco=0, cs=1, oe=0) # CS=1 bloqueia
dram4.escrever(10, banco=0, dado=0xC, we=1) # WE=1 bloqueia

print("\nTeste concluído com sucesso.")

=== ATIVIDADE 2.5 - MEMÓRIA 3: DRAM 128M x 4, 2 BANCOS ===

Chip inicializado: 2 bancos, 8192x8192 cada
Total: 134217728 posições de 4 bits = 64 MBytes = 512 Mbits

Testando escrita de nibbles:
ESCRITA: Banco=0 | End=00000000 | Lin=0000 | Col=0000 | D=A | 1010
ESCRITA: Banco=0 | End=00008192 | Lin=0001 | Col=0000 | D=F | 1111
ESCRITA: Banco=1 | End=00000000 | Lin=0000 | Col=0000 | D=5 | 0101
ESCRITA: Banco=1 | End=134217727 | Lin=8191 | Col=8191 | D=3 | 0011

Testando leitura:
LEITURA: Banco=0 | End=00000000 | Lin=0000 | Col=0000 | D=A | 1010
LEITURA: Banco=0 | End=00008192 | Lin=0001 | Col=0000 | D=F | 1111
LEITURA: Banco=1 | End=00000000 | Lin=0000 | Col=0000 | D=5 | 0101
LEITURA: Banco=1 | End=134217727 | Lin=8191 | Col=8191 | D=3 | 0011
LEITURA: Banco=0 | End=00000100 | Lin=0000 | Col=0100 | D=0 | 0000

Teste de sinais de controle:
LEITURA BLOQUEADA: CS=1 OE=0
ESCRITA BLOQUEADA: CS=0 WE=1

Teste concluído com sucesso.
